# IT2011 - Artificial Intelligence and Machine Learning
## Progress Review I: Data Preprocessing & Exploratory Data Analysis (EDA)
### Group ID: `2026-Y2-S1-MET-23`
### Member 4: Abdullah H.F. (IT25102877)
### Assigned Technique: Numerical Cleaning, Outlier Handling & Feature Scaling

---
### 1. Technique Overview & Academic Justification
The dataset contains numerical signals such as user `Ratings` (1.0 to 10.0) and text length statistics (review length, word count):
1. **Outlier Impact:** In online user reviews, extreme outliers exist (e.g. 5,000-word essays vs. 2-word comments). If fed directly to machine learning algorithms, extreme values skew distance metrics (Euclidean distances in SVM/kNN) and destabilize gradient descent optimization.
2. **IQR-Based Winsorization (Clipping):** Rather than deleting rows and losing valuable emotional annotations, we employ IQR-based capping:
   $$\text{Upper Fence} = Q_3 + 1.5 \times \text{IQR}, \quad \text{Lower Fence} = Q_1 - 1.5 \times \text{IQR}$$
   Extreme values are safely capped at the boundary fences.
3. **Feature Scaling:** `Ratings` ranges from 1 to 10, whereas review length ranges into the hundreds. We apply `RobustScaler` and `MinMaxScaler` to bring features onto commensurate scales without allowing magnitude to dominate model weights.

**Viva Objective:** Demonstrate mathematical outlier bounds, explain why clipping preserves data integrity, and visualize ratings distribution across emotion classes.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler, MinMaxScaler

os.makedirs('../results/eda_visualizations', exist_ok=True)
sns.set_theme(style="whitegrid", palette="muted")


### 2. Loading the Raw Dataset & Extracting Numerical Features

In [ ]:
DATA_PATH = '../data/raw/Movies_Reviews_modified_version1.csv'
df = pd.read_csv(DATA_PATH)

# Extract numeric meta-features
df['word_count'] = df['Reviews'].astype(str).apply(lambda x: len(x.split()))
df['char_count'] = df['Reviews'].astype(str).apply(len)

print("Numerical Summary Statistics:")
df[['Ratings', 'word_count', 'char_count']].describe().round(2)


### 3. Outlier Detection via Interquartile Range (IQR) & Winsorization

In [ ]:
def detect_and_cap_iqr(series: pd.Series, factor: float = 1.5):
    """Calculates IQR boundaries and applies winsorization (clipping)."""
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = max(0, q1 - factor * iqr)
    upper_bound = q3 + factor * iqr
    
    outlier_mask = (series < lower_bound) | (series > upper_bound)
    capped_series = series.clip(lower=lower_bound, upper=upper_bound)
    
    return capped_series, outlier_mask.sum(), lower_bound, upper_bound

# Detect and cap word_count outliers
df['word_count_capped'], wc_outliers, wc_low, wc_high = detect_and_cap_iqr(df['word_count'])
print(f"Word Count: {wc_outliers:,} outliers detected. Bounds: [{wc_low:.1f}, {wc_high:.1f}]")

# Check Ratings (Valid range: 1.0 - 10.0)
print("Ratings minimum:", df['Ratings'].min(), "| maximum:", df['Ratings'].max())


### 4. Robust Scaling & Normalization Pipeline

In [ ]:
# Apply RobustScaler (scales based on median and IQR, resistant to residual outliers)
robust_scaler = RobustScaler()
df[['word_count_scaled', 'Ratings_scaled']] = robust_scaler.fit_transform(df[['word_count_capped', 'Ratings']])

# Apply MinMaxScaler to map ratings into [0, 1] for neural models
minmax_scaler = MinMaxScaler()
df['Ratings_norm'] = minmax_scaler.fit_transform(df[['Ratings']])

df[['Ratings', 'Ratings_norm', 'word_count', 'word_count_capped', 'word_count_scaled']].head()


### 5. Individual EDA Visualizations (Viva Presentation Requirement)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left Plot: Boxplot of Word Counts before and after capping
box_data = pd.DataFrame({
    'Raw Word Count': df['word_count'],
    'Capped Word Count': df['word_count_capped']
})
sns.boxplot(data=box_data, palette=['#e74c3c', '#2ecc71'], ax=axes[0])
axes[0].set_title('Review Word Count: Raw vs. IQR Capped Outliers', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Word Count', fontsize=11)

# Right Plot: Ratings distribution across emotion classes
sns.violinplot(data=df, x='emotion', y='Ratings', palette='magma', ax=axes[1], inner='quartile')
axes[1].set_title('User Ratings Distribution by Emotion Category', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Target Emotion', fontsize=11)
axes[1].set_ylabel('Numerical Rating (1 - 10)', fontsize=11)
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
output_plot_path = '../results/eda_visualizations/member4_numerical_outliers_and_ratings.png'
plt.savefig(output_plot_path, dpi=300, bbox_inches='tight')
print(f"EDA plot saved successfully to: {output_plot_path}")
plt.show()


### 6. Key Findings & Viva Talking Points (For Abdullah H.F.)

> **Viva Preparation Notes:**
> 1. **Why did you choose Capping (Winsorization) over Outlier Deletion?**
>    Deleting rows with long reviews would discard thousands of authentic movie reviews and shrink minority emotion classes. Capping preserves 100% of data points while restraining disproportionate gradient updates.
> 2. **Why is scaling necessary when combining text embeddings with numeric features?**
>    Word counts can reach values above 500, while `Ratings` is bounded between 1 and 10. Without scaling, unscaled word counts would artificially dominate distance-based and linear algorithms.
> 3. **What does the violin plot reveal?**
>    There is a profound relationship between `Ratings` and `emotion`: `joy` and `optimism` have median ratings around 8-10, whereas `disgust` and `anger` peak sharply at 1-3. This confirms `Ratings` is a high-yield feature.
